In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../..").resolve()))

import pandas as pd
from src.utils.db import get_connection

conn = get_connection()

df_reviews = pd.read_sql(
    "SELECT * FROM steam_indie_reviews",
    conn
)

df_list = pd.read_sql(
    "SELECT * FROM steam_indie_9692",
    conn
)

df_details = pd.read_sql(
    "SELECT * FROM steam_app_details",
    conn
)

conn.close()

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_1740\3153904167.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_reviews = pd.read_sql(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_1740\3153904167.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_list = pd.read_sql(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_1740\3153904167.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_details = pd.read_sql(


In [6]:
df_details.dtypes

appid                    int64
name                       str
type                       str
is_free                    str
controller_support         str
short_description          str
supported_languages        str
developers                 str
publishers                 str
genres                     str
categories                 str
coming_soon                str
release_date               str
currency                   str
initial                    str
final                      str
discount_percent           str
initial_formatted          str
final_formatted            str
windows                    str
mac                        str
linux                      str
recommendations_total      str
metacritic_score           str
metacritic_url             str
achievements_total         str
header_image               str
website                    str
collected_at               str
dtype: object

In [32]:
import datetime
df_reviews['timestamp_created'] = df_reviews['timestamp_created'].astype('int64')
df_reviews['timestamp_created'] = df_reviews['timestamp_created'].apply(lambda x: datetime.datetime.fromtimestamp(x))
df_reviews['timestamp_created'] = df_reviews['timestamp_created'].dt.tz_localize('UTC').dt.tz_convert('Asia/Seoul')
df_reviews['timestamp_created']

0       2023-03-13 11:48:50+09:00
1       2023-03-29 09:50:29+09:00
2       2023-04-11 11:56:57+09:00
3       2023-05-26 11:07:39+09:00
4       2023-04-17 06:50:31+09:00
                   ...           
13101   2023-12-23 08:39:35+09:00
13102   2023-12-22 15:54:43+09:00
13103   2023-12-22 15:46:45+09:00
13104   2023-12-22 14:14:37+09:00
13105   2023-12-22 11:28:24+09:00
Name: timestamp_created, Length: 13106, dtype: datetime64[us, Asia/Seoul]

In [33]:
df_reviews['timestamp_updated'] = df_reviews['timestamp_updated'].astype('int64')
df_reviews['timestamp_updated'] = df_reviews['timestamp_updated'].apply(lambda x: datetime.datetime.fromtimestamp(x))
df_reviews['timestamp_updated'] = df_reviews['timestamp_updated'].dt.tz_localize('UTC').dt.tz_convert('Asia/Seoul')
df_reviews['timestamp_updated']

0       2023-03-13 11:48:50+09:00
1       2023-03-31 10:03:31+09:00
2       2023-04-11 11:56:57+09:00
3       2023-05-26 11:07:39+09:00
4       2023-04-23 03:01:56+09:00
                   ...           
13101   2025-05-23 21:34:50+09:00
13102   2023-12-22 15:54:43+09:00
13103   2023-12-22 15:46:45+09:00
13104   2025-03-03 21:17:10+09:00
13105   2024-03-09 16:09:25+09:00
Name: timestamp_updated, Length: 13106, dtype: datetime64[us, Asia/Seoul]

In [34]:
TARGET_STRATA = ['large_high', 'mid_high', 'small_high']
df_high = df_list[df_list['stratum'].isin(TARGET_STRATA)].reset_index(drop=True)
df_high['stratum'].value_counts()

stratum
large_high    29
mid_high      25
small_high    20
Name: count, dtype: int64

In [35]:
df_high['positive'] = df_high['positive'].astype("int64")
df_high['negative'] = df_high['negative'].astype("int64")
df_high['total_reviews'] = df_high['total_reviews'].astype("int64")

In [6]:
from pprint import pprint
app_id = df_high['appid'].to_list()

games = []
for id in app_id:
    game = df_high[df_high["appid"] == id]
    games.append({
        "app_id" : id,
        "name" : game['name_store'].item(),
        # 전체 리뷰수 대비 긍정 비율
        "positive_rate" : round(game['positive'].item() / game["total_reviews"].item() * 100, 2),
        # 부정 대비 긍정율
        "positive_score" : game['positive'].item() / game['negative'].item(),
        "group" : game['stratum'].item()
    })

df_status = pd.DataFrame(games)
df_status

,app_id,name,positive_rate,positive_score,group
0,1432860,Sun Haven,82.49,4.709636,large_high
1,1473350,(the) Gnorp Apologue,96.39,26.697279,large_high
2,1993150,轮回修仙路,82.20,4.617647,large_high
3,2527500,MiSide,98.02,49.402450,large_high
4,1169040,Necesse,93.66,14.762696,large_high
...,...,...,...,...,...
69,2729480,NightClub Simulator,93.82,15.178571,small_high
70,2868430,XiuzhenWorld,76.78,3.306931,small_high
71,1000440,东方雪莲华 ～ Abyss Soul Lotus.,92.51,12.343750,small_high
72,2178590,Coin Pusher Casino,85.23,5.768519,small_high


In [7]:
df_lh = df_status[df_status['group'] == 'large_high']
df_mh = df_status[df_status['group'] == 'mid_high']
df_sh = df_status[df_status['group'] == 'small_high']

In [8]:
df = df_reviews.copy()

df['author_playtime_at_review'] = df['author_playtime_at_review'].astype('int64')

In [9]:
sh_games = df_sh['app_id'].to_list()
mh_games = df_mh['app_id'].to_list()
lh_games = df_lh['app_id'].to_list()

In [10]:
review_lh = df[df['appid'].isin(lh_games)]
review_mh = df[df['appid'].isin(mh_games)]
review_sh = df[df['appid'].isin(sh_games)]

review_lh.head()

,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,steam_purchase,received_for_free,written_during_early_access,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played
0,134554560,1432860,english,"For me, Sun Haven is a fun farming game that h...",2023-03-13 11:48:50+09:00,2023-03-13 11:48:50+09:00,True,1432,25,0.971922159194946289,...,True,False,False,76561199471119493,0,1,4120,0,1723,1763967370
1,135610350,1432860,english,*Updated for v1.0.3*\n\nI really love this gam...,2023-03-29 09:50:29+09:00,2023-03-31 10:03:31+09:00,False,1564,18,0.939695596694946289,...,True,False,False,76561198107464933,663,108,12951,0,8605,1690050219
2,136483272,1432860,english,(Singleplayer review)\nPuh i am not really sur...,2023-04-11 11:56:57+09:00,2023-04-11 11:56:57+09:00,True,166,0,0.917131006717681885,...,True,False,False,76561198005466151,1021,4,3653,0,3400,1723728962
3,138973732,1432860,russian,"Я наиграла в эту игру почти 55 часов, 40+ из к...",2023-05-26 11:07:39+09:00,2023-05-26 11:07:39+09:00,False,224,7,0.897390604019165039,...,True,False,False,76561198098873026,720,16,3280,0,3280,1684534023
4,136801481,1432860,japanese,Sun Haven好きさんが増えると嬉しいので、初めてレビューします。\n\nこれを見たあな...,2023-04-17 06:50:31+09:00,2023-04-23 03:01:56+09:00,True,143,10,0.895800650119781494,...,True,False,False,76561199151368353,0,1,12057,0,7849,1723475976


In [11]:
# large_high부터 테스트
# 1. 긍정/부정 가독성을 위해 데이터 라벨링
review_lh['sentiment'] = review_lh['voted_up'].str.lower().map({'true': True, 'false': False})

# 2. 사분위수(4등분)를 기준으로 리뷰 생성 시점에서 플레이타임 구간 생성
review_lh['playtime_group'] = pd.qcut(review_lh['author_playtime_at_review'], 
                            q=4, 
                            labels=['하위 25%(라이트)', '중하위', '중상위', '상위 25%(헤비)'])

review_lh['playtime_group'].value_counts()

playtime_group
하위 25%(라이트)    1386
중상위            1382
상위 25%(헤비)     1382
중하위            1380
Name: count, dtype: int64

In [12]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# plt.rcParams['font.family'] = 'AppleGothic'    # Mac
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

# 3. 시각화 (Stacked Bar Chart)
fig = px.bar(review_lh.groupby(['playtime_group', 'sentiment']).size().reset_index(name='count'), 
            x="playtime_group", 
            y="count", 
            color="sentiment",
            barmode="stack",
            title = "large_high 그룹의 플레이타임 구간 별 긍정/부정 비율") # 일단 쌓기 모드로 생성

fig.show()

In [13]:
# mid_high 테스트
# 1. 긍정/부정 가독성을 위해 데이터 라벨링
review_mh['sentiment'] = review_mh['voted_up'].str.lower().map({'true': True, 'false': False})

# 2. 사분위수(4등분)를 기준으로 리뷰 생성 시점에서 플레이타임 구간 생성
review_mh['playtime_group'] = pd.qcut(review_mh['author_playtime_at_review'], 
                            q=4, 
                            labels=['하위 25%(라이트)', '중하위', '중상위', '상위 25%(헤비)'])

review_mh['playtime_group'].value_counts()

playtime_group
중하위            1106
하위 25%(라이트)    1098
상위 25%(헤비)     1098
중상위            1090
Name: count, dtype: int64

In [14]:
# 3. 시각화 (Stacked Bar Chart)
fig = px.bar(review_lh.groupby(['playtime_group', 'sentiment']).size().reset_index(name='count'), 
            x="playtime_group", 
            y="count", 
            color="sentiment",
            barmode="stack",
            title = "mid_high 그룹의 플레이타임 구간 별 긍정/부정 비율") # 일단 쌓기 모드로 생성

fig.show()

In [14]:
# small_high 테스트
# 1. 긍정/부정 가독성을 위해 데이터 라벨링
review_sh['sentiment'] = review_sh['voted_up'].str.lower().map({'true': True, 'false': False})

# 2. 사분위수(4등분)를 기준으로 리뷰 생성 시점에서 플레이타임 구간 생성
review_sh['playtime_group'] = pd.qcut(review_sh['author_playtime_at_review'], 
                            q=4, 
                            labels=['하위 25%(라이트)', '중하위', '중상위', '상위 25%(헤비)'])

review_sh['playtime_group'].value_counts()

playtime_group
하위 25%(라이트)    808
중상위            796
상위 25%(헤비)     796
중하위            784
Name: count, dtype: int64

In [16]:
# 3. 시각화 (Stacked Bar Chart)
fig = px.bar(review_sh.groupby(['playtime_group', 'sentiment']).size().reset_index(name='count'), 
            x="playtime_group", 
            y="count", 
            color="sentiment",
            barmode="stack",
            title = "small_high 그룹의 플레이타임 구간 별 긍정/부정 비율") # 일단 쌓기 모드로 생성

fig.show()

In [15]:
target_genres = ["Action", "Casual", "Adventure", "Simulation", "Strategy", "RPG", "Racing", "Sports"]

for genre in target_genres:
    cnt = df_list['genres'].apply(lambda x: genre in x).sum()
    print(f"{genre}: {cnt}개")

Action: 75개
Casual: 58개
Adventure: 82개
Simulation: 44개
Strategy: 41개
RPG: 46개
Racing: 5개
Sports: 9개


In [16]:
adventure = df_list[df_list['genres'].apply(lambda x: 'Adventure' in x)]
action = df_list[df_list['genres'].apply(lambda x: 'Action' in x)]

In [17]:
adventure_list = adventure['appid'].to_list()
action_list = action['appid'].to_list()
review_ad = df[df['appid'].isin(adventure_list)]
review_ac = df[df['appid'].isin(action_list)]

In [18]:
review_ad['sentiment'] = review_ad['voted_up'].str.lower().map({'true': True, 'false': False})

# 2. 사분위수(4등분)를 기준으로 리뷰 생성 시점에서 플레이타임 구간 생성
review_ad['playtime_group'] = pd.qcut(review_ad['author_playtime_at_review'], 
                            q=4, 
                            labels=['하위 25%(라이트)', '중하위', '중상위', '상위 25%(헤비)'])

review_ad['playtime_group'].value_counts()

playtime_group
하위 25%(라이트)    1588
중상위            1582
중하위            1580
상위 25%(헤비)     1580
Name: count, dtype: int64

In [21]:
# 3. 시각화 (Stacked Bar Chart)
fig = px.bar(review_ad.groupby(['playtime_group', 'sentiment']).size().reset_index(name='count'), 
            x="playtime_group", 
            y="count", 
            color="sentiment",
            barmode="stack",
            title = "어드벤쳐 장르 게임의 플레이타임 구간 별 긍정/부정 비율") # 일단 쌓기 모드로 생성

fig.show()

In [19]:
review_ac['sentiment'] = review_ac['voted_up'].str.lower().map({'true': True, 'false': False})

# 2. 사분위수(4등분)를 기준으로 리뷰 생성 시점에서 플레이타임 구간 생성
review_ac['playtime_group'] = pd.qcut(review_ac['author_playtime_at_review'], 
                            q=4, 
                            labels=['하위 25%(라이트)', '중하위', '중상위', '상위 25%(헤비)'])

review_ac['playtime_group'].value_counts()

playtime_group
하위 25%(라이트)    1254
중하위            1252
상위 25%(헤비)     1252
중상위            1248
Name: count, dtype: int64

In [23]:
# 3. 시각화 (Stacked Bar Chart)
fig = px.bar(review_ac.groupby(['playtime_group', 'sentiment']).size().reset_index(name='count'), 
            x="playtime_group", 
            y="count", 
            color="sentiment",
            barmode="stack",
            title = "액션 장르 게임의 플레이타임 구간 별 긍정/부정 비율") # 일단 쌓기 모드로 생성

fig.show()

In [20]:
df_list_2 = df_list.copy()
df_list_2['total_reviews'] = df_list_2['total_reviews'].astype('int64')
df_list_2 = df_list_2.sort_values('total_reviews', ascending = False)
df_list_2

,appid,name_store,release_date,genres,owners,owners_lower,positive,negative,total_reviews,price_spy,ccu,developers,stratum,is_f2p
28,2379780,Balatro,2024-02-20,"['Casual', 'Indie', 'Strategy']","2,000,000 .. 5,000,000",2000000,150524,3042,153566,1499,17123,LocalThunk,large_high,False
3,2527500,MiSide,2024-12-10,"['Adventure', 'Indie', 'RPG', 'Simulation']","1,000,000 .. 2,000,000",1000000,108883,2204,111087,1499,631,AIHASTO,large_high,False
17,1049590,Eternal Return,2023-07-19,"['Indie', 'Strategy', 'Free To Play']","5,000,000 .. 10,000,000",5000000,51694,13352,65046,0,12341,Nimble Neuron,large_high,True
0,1432860,Sun Haven,2023-03-10,"['Adventure', 'Casual', 'Indie', 'RPG', 'Simul...","500,000 .. 1,000,000",500000,18523,3933,22456,2499,653,Pixel Sprout Studios,large_high,False
11,2198150,Tiny Glade,2024-09-23,"['Casual', 'Indie', 'Simulation']","500,000 .. 1,000,000",500000,19362,483,19845,1499,291,Pounce Light,large_high,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44,2230410,Eaten by Darkness,2023-04-01,"['Action', 'Indie']","500,000 .. 1,000,000",500000,3,7,10,99,0,Black Cardioid Games,large_low,False
46,3217030,Moxter,2024-12-06,"['Adventure', 'Indie', 'Racing', 'Sports']","200,000 .. 500,000",200000,7,3,10,999,0,"Aisten Lab, Dalton Bastos",large_low,False
106,2222480,Clash: Robot Detective - Complete Edition,2023-01-02,"['Adventure', 'Indie']","20,000 .. 50,000",20000,10,0,10,499,0,Drone Garden Studios,mid_low,False
156,3061550,RIOO,2024-08-20,"['Adventure', 'Casual', 'Indie']","0 .. 20,000",0,8,2,10,99,1,Project Ground Zero,small_low,False


In [21]:
id_list = df_list_2['appid'].to_list()
id_1 = id_list[0]
id_2 = id_list[1]

In [22]:
reviews_1 = df[df['appid'] == id_1]
reviews_1['sentiment'] = reviews_1['voted_up'].str.lower().map({'true': True, 'false': False})

# 2. 사분위수(4등분)를 기준으로 리뷰 생성 시점에서 플레이타임 구간 생성
reviews_1['playtime_group'] = pd.qcut(reviews_1['author_playtime_at_review'], 
                            q=4, 
                            labels=['하위 25%(라이트)', '중하위', '중상위', '상위 25%(헤비)'])

reviews_1['playtime_group'].value_counts()

playtime_group
하위 25%(라이트)    50
중하위            50
중상위            50
상위 25%(헤비)     50
Name: count, dtype: int64

In [27]:
# 3. 시각화 (Stacked Bar Chart)
fig = px.bar(reviews_1.groupby(['playtime_group', 'sentiment']).size().reset_index(name='count'), 
            x="playtime_group", 
            y="count", 
            color="sentiment",
            barmode="stack",
            title = f"ID = {id_1} 게임의 플레이타임 구간 별 긍정/부정 비율") # 일단 쌓기 모드로 생성

fig.show()

In [23]:
reviews_2 = df[df['appid'] == id_2]
reviews_2['sentiment'] = reviews_2['voted_up'].str.lower().map({'true': True, 'false': False})

# 2. 사분위수(4등분)를 기준으로 리뷰 생성 시점에서 플레이타임 구간 생성
reviews_2['playtime_group'] = pd.qcut(reviews_2['author_playtime_at_review'], 
                            q=4, 
                            labels=['하위 25%(라이트)', '중하위', '중상위', '상위 25%(헤비)'])

reviews_2['playtime_group'].value_counts()

playtime_group
하위 25%(라이트)    50
중하위            50
중상위            50
상위 25%(헤비)     50
Name: count, dtype: int64

In [29]:
# 3. 시각화 (Stacked Bar Chart)
fig = px.bar(reviews_2.groupby(['playtime_group', 'sentiment']).size().reset_index(name='count'), 
            x="playtime_group", 
            y="count", 
            color="sentiment",
            barmode="stack",
            title = f"ID = {id_2} 게임의 플레이타임 구간 별 긍정/부정 비율") # 일단 쌓기 모드로 생성

fig.show()

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import ast

# 데이터 로드 (파일 경로에 맞춰 수정)
# df = pd.read_csv('indie_games_details.csv')

df_merge = pd.merge(
    df_list,
    df_details[['appid', 'controller_support', 'publishers', 'categories', 'coming_soon', 'windows', 'mac', 'linux', 'recommendations_total', 'achievements_total']],
    on = 'appid'
)

df_merge['positive'] = df_merge['positive'].astype('int64')
df_merge['negative'] = df_merge['negative'].astype('int64')

# 1. Stratum(그룹)별 검정용 빈도표 생성
stratum_table = df_merge.groupby('stratum')[['positive', 'negative']].sum()

# 2. 장르별 검정용 데이터 준비 (장르 확장)
df_merge['genres_list'] = df_merge['genres'].apply(ast.literal_eval)
df_genre = df_merge.explode('genres_list')
target_genres = ["Action", "RPG", "Simulation", "Strategy", "Adventure"]
df_genre_filtered = df_genre[df_genre['genres_list'].isin(target_genres)]

genre_table = df_genre_filtered.groupby('genres_list')[['positive', 'negative']].sum()

print("--- Stratum별 빈도표 ---")
print(stratum_table)

print("--- 장르별 빈도표 ---")
print(genre_table)

--- Stratum별 빈도표 ---
            positive  negative
stratum                       
large_high    493395     41194
large_low        160        25
large_mid       2518       521
mid_high       24309      3617
mid_low          127        34
mid_mid         3255       808
small_high      8627      1697
small_low        201        62
small_mid       1752       275
--- 장르별 빈도표 ---
             positive  negative
genres_list                    
Action          97688     14327
Adventure      229387     19723
RPG            166834     11153
Simulation     216866     15139
Strategy       254072     24951


In [40]:
def run_chi2(table, name):
    # 카이제곱 검정 수행
    chi2, p, dof, expected = chi2_contingency(table)
    
    print(f"\n[{name} - 긍부정 관계 검정 결과]")
    print(f"카이제곱 통계량: {chi2:.4f}")
    print(f"p-value: {p:.4e}")
    
    if p < 0.05:
        print(f"결과: 통계적으로 유의미한 차이가 발견되었습니다. (p < 0.05)")
        print(f"해석: {name}에 따라 유저들의 긍부정 평가 비율이 다릅니다.")
    else:
        print(f"결과: 통계적으로 유의미한 차이가 없습니다. (p >= 0.05)")
        print(f"해석: {name}에 상관없이 긍부정 평가 비율은 독립적입니다.")

# 실행
run_chi2(stratum_table, "Stratum 그룹")
run_chi2(genre_table, "장르")


[Stratum 그룹 - 긍부정 관계 검정 결과]
카이제곱 통계량: 3172.0302
p-value: 0.0000e+00
결과: 통계적으로 유의미한 차이가 발견되었습니다. (p < 0.05)
해석: Stratum 그룹에 따라 유저들의 긍부정 평가 비율이 다릅니다.

[장르 - 긍부정 관계 검정 결과]
카이제곱 통계량: 5150.8370
p-value: 0.0000e+00
결과: 통계적으로 유의미한 차이가 발견되었습니다. (p < 0.05)
해석: 장르에 따라 유저들의 긍부정 평가 비율이 다릅니다.
